# CS401 — Multimodal Data Management with SurrealDB
### Demo Notebook: Parsing, Ingestion, and Queries

This notebook walks through all four project phases:
1. Data Parsing
2. Schema & SurrealDB Setup
3. Ingestion
4. Multimodal Queries

In [1]:
import sys, json
from pathlib import Path

# Make sure project root is on path
sys.path.insert(0, str(Path("..").resolve()))

## Phase 1 — Data Parsing

### 1a. Parse Lecture Notes (PDF)

In [5]:
from parsers.pdf_parser import parse_pdf

lecture = parse_pdf("../dataset/text/lecture_notes.pdf")

# Print everything except the full content body
summary = {k: v for k, v in lecture.items() if k != "content"}
print(json.dumps(summary, indent=2))
print(f"\nContent preview (first 400 chars):\n{lecture.get('content','')[:400]}")

{
  "id": "lecture:lecture_notes",
  "type": "lecture",
  "source": "..\\dataset\\text\\lecture_notes.pdf",
  "filename": "lecture_notes.pdf",
  "title": "Advanced Database Systems \u2014 Lecture Notes",
  "page_count": 1,
  "topics": []
}

Content preview (first 400 chars):
Advanced Database Systems — Lecture Notes
CS401 | Week 1 | Dr. Sarah Chen
1. Normalization
Database normalization is the process of structuring a relational database to reduce data redundancy
and improve data integrity. The primary normal forms are 1NF, 2NF, and 3NF. A relation is in 1NF if all
attribute values are atomic. A relation is in 2NF if it is in 1NF and every non-key attribute is fully
f


### 1b. Parse Assignment (TXT)

In [6]:
from parsers.txt_parser import parse_txt

assignment = parse_txt("../dataset/text/assignment.txt")
print(json.dumps({k: v for k, v in assignment.items() if k != "content"}, indent=2))
print(f"\nContent preview:\n{assignment['content'][:300]}")

{
  "id": "assignment:assignment",
  "type": "assignment",
  "source": "..\\dataset\\text\\assignment.txt",
  "filename": "assignment.txt",
  "title": "Assignment 1: Database Normalization",
  "word_count": 104,
  "concepts": [
    "Normalization",
    "Functional Dependencies",
    "Primary Keys",
    "Foreign Keys",
    "Indexes",
    "Query Optimization"
  ],
  "due_date": "Week 3"
}

Content preview:
Assignment 1: Database Normalization

Due Date: Week 3

Objectives:
- Understand the principles of database normalization (1NF, 2NF, 3NF)
- Apply normalization techniques to a real-world dataset
- Evaluate trade-offs between normalized and denormalized schemas

Tasks:
1. Given the student enrollment


### 1c. Parse Student Scores (CSV)

In [7]:
from parsers.csv_parser import parse_csv

csv_data = parse_csv("../dataset/tables/student_scores.csv")
print(f"Columns : {csv_data['columns']}")
print(f"Rows    : {csv_data['row_count']}")
print("\nFirst 3 records:")
print(json.dumps(csv_data["records"][:3], indent=2))

Columns : ['student_id', 'name', 'assignment_1', 'assignment_2', 'midterm', 'final', 'grade']
Rows    : 10

First 3 records:
[
  {
    "student_id": "S001",
    "name": "Alice Johnson",
    "assignment_1": 92,
    "assignment_2": 88,
    "midterm": 85,
    "final": 91,
    "grade": "A",
    "id": "student_score:S001",
    "source": "..\\dataset\\tables\\student_scores.csv",
    "type": "student_score"
  },
  {
    "student_id": "S002",
    "name": "Bob Smith",
    "assignment_1": 78,
    "assignment_2": 82,
    "midterm": 74,
    "final": 79,
    "grade": "B+",
    "id": "student_score:S002",
    "source": "..\\dataset\\tables\\student_scores.csv",
    "type": "student_score"
  },
  {
    "student_id": "S003",
    "name": "Carol White",
    "assignment_1": 95,
    "assignment_2": 97,
    "midterm": 93,
    "final": 96,
    "grade": "A+",
    "id": "student_score:S003",
    "source": "..\\dataset\\tables\\student_scores.csv",
    "type": "student_score"
  }
]


### 1d. Parse JSON Metadata

In [9]:
from parsers.json_parser import parse_json_metadata

metadata = parse_json_metadata("../dataset/metadata/metadata.json")
print(f"Valid    : {metadata['valid']}")
print(f"Warnings : {metadata['warnings']}")
print(f"Course   : {metadata['data']['course']['title']}")
print(f"Materials: {metadata['material_ids']}")

Valid    : True
Warnings : []
Course   : Advanced Database Systems
Materials: ['mat001', 'mat002', 'mat003', 'mat004']


### 1e. Parse Images

In [10]:
from parsers.image_parser import parse_images_in_directory

images = parse_images_in_directory("../dataset/images")
print(json.dumps(images, indent=2))

[
  {
    "id": "image:sample_plot",
    "type": "image",
    "source": "..\\dataset\\images\\sample_plot.png",
    "filename": "sample_plot.png",
    "format": "PNG",
    "file_size_bytes": 5583,
    "width": 600,
    "height": 400,
    "mode": "RGB"
  }
]


## Phase 2 — Schema Design

The schema is defined in `schema.surql`. Key design decisions:

| Table | Type | Rationale |
|-------|------|-----------|
| `lecture` | SCHEMAFULL | Consistent PDF-derived fields; BM25 FTS index |
| `assignment` | SCHEMAFULL | Consistent TXT-derived fields; BM25 FTS index |
| `student_score` | SCHEMAFULL | Typed numeric fields; UNIQUE student_id index |
| `metadata` | SCHEMALESS | Flexible JSON document; shape varies per course |
| `image` | SCHEMAFULL | File metadata; optional width/height |
| `concept` | SCHEMAFULL | Graph nodes representing abstract topics |

**Graph edges:** `covers`, `requires`, `describes`, `submitted_for`

In [12]:
with open("../schema.surql") as f:
    schema_text = f.read()
print(schema_text[:2000], "…")

-- ============================================================
-- schema.surql
-- SurrealDB multimodal schema for CS401 Advanced Database Systems
-- ============================================================
-- Run with:
--   surreal import --conn http://localhost:8000 \
--                  --user root --pass root \
--                  --ns education --db cs401 \
--                  schema.surql
-- ============================================================


-- â”€â”€ Namespace & Database â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
DEFINE NAMESPACE education;
DEFINE DATABASE cs401;

USE NS education DB cs401;


-- â”€â”€ TABLE: lecture â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
-- Stores parsed PDF lecture notes.
DEFINE TABLE lecture SCHEMAFULL;

DEFINE FIELD id          ON lecture TYPE string;
DEFINE FIELD title       ON 

## Phase 3 — Ingestion

The ingestion script handles the full pipeline in one command:
```bash
python -m ingestion.ingest --http
```
Below we demonstrate a dry run (no DB connection required).

In [13]:
from ingestion.ingest import run as ingest_all

ingest_all(dry_run=True)

[ingest] === Multimodal Ingestion Pipeline ===
[ingest] --- Phase 1: Parsing ---
[ingest] ✓ Parsed PDF lecture notes
[ingest] ✓ Parsed TXT assignment
[ingest] ✓ Parsed CSV scores
[ingest] ✓ Parsed JSON metadata
[ingest] ✓ Parsed Images directory
[ingest] --- DRY RUN: Printing parsed records ---

  LECTURE
{
  "id": "lecture:lecture_notes",
  "type": "lecture",
  "source": "D:\\Downloads\\database\\surrealdb_project\\dataset\\text\\lecture_notes.pdf",
  "filename": "lecture_notes.pdf",
  "title": "Advanced Database Systems \u2014 Lecture Notes",
  "page_count": 1,
  "content": "Advanced Database Systems \u2014 Lecture Notes\nCS401 | Week 1 | Dr. Sarah Chen\n1. Normalization\nDatabase normalization is the process of structuring a relational database to reduce data redundancy\nand impr\u2026",
  "topics": [
    "Normalization",
    "Indexes",
    "Query Optimization",
    "Transactions"
  ]
}

  ASSIGNMENT
{
  "id": "assignment:assignment",
  "type": "assignment",
  "source": "D:\\Downloa

## Phase 4 — Multimodal Queries

> **Note:** The cells below require a running SurrealDB instance.
> Start one with:
> ```bash
> surreal start --user root --pass root memory
> ```
> Then re-run: `ingest_all(use_http=True)` to populate the DB.

In [14]:
# ── Query 1: Full-Text Keyword Search ──────────────────────────────────────────
QUERY_1 = """
USE NS education DB cs401;
LET $keyword = 'normalization';
SELECT id, title, type, search::score(1) AS relevance_score
FROM lecture WHERE content @1@ $keyword
UNION ALL
SELECT id, title, type, search::score(1) AS relevance_score
FROM assignment WHERE content @1@ $keyword
ORDER BY relevance_score DESC;
"""
print("Query 1: Full-Text Keyword Search")
print(QUERY_1)

Query 1: Full-Text Keyword Search

USE NS education DB cs401;
LET $keyword = 'normalization';
SELECT id, title, type, search::score(1) AS relevance_score
FROM lecture WHERE content @1@ $keyword
UNION ALL
SELECT id, title, type, search::score(1) AS relevance_score
FROM assignment WHERE content @1@ $keyword
ORDER BY relevance_score DESC;



In [ ]:
# ── Query 2: Student Scores with Average ───────────────────────────────────────
QUERY_2 = """
USE NS education DB cs401;
SELECT
    student_id, name,
    assignment_1, assignment_2, midterm, final, grade,
    math::mean([assignment_1, assignment_2, midterm, final]) AS avg_score
FROM student_score
ORDER BY avg_score DESC;
"""
print("Query 2: Student Scores Join")
print(QUERY_2)

In [ ]:
# ── Query 3: Lecture ↔ Metadata Topic Linkage ──────────────────────────────────
QUERY_3 = """
USE NS education DB cs401;
SELECT id, title, topics, page_count, source FROM lecture;
"""
print("Query 3: Lecture Content Linked to Metadata")
print(QUERY_3)

In [ ]:
# ── Query 4: Graph Traversal ───────────────────────────────────────────────────
QUERY_4 = """
USE NS education DB cs401;
SELECT
    id AS concept_id,
    name AS concept_name,
    <-covers<-lecture.title AS covered_by_lectures,
    <-requires<-assignment.title AS required_by_assignments
FROM concept
ORDER BY concept_name;
"""
print("Query 4: Graph Traversal — Shared Concepts")
print(QUERY_4)

In [ ]:
# ── Query 5: Grade Distribution Summary ───────────────────────────────────────
QUERY_5 = """
USE NS education DB cs401;
SELECT
    grade,
    count() AS num_students,
    math::mean(final) AS avg_final,
    math::min(final) AS min_final,
    math::max(final) AS max_final
FROM student_score
GROUP BY grade
ORDER BY avg_final DESC;
"""
print("Query 5: Grade Distribution")
print(QUERY_5)

## Summary Statistics from Parsed Data
(No DB required — computed from parsed CSV)

In [ ]:
import statistics

scores = csv_data["records"]
finals = [r["final"] for r in scores]
avgs   = [round((r["assignment_1"] + r["assignment_2"] + r["midterm"] + r["final"]) / 4, 2)
          for r in scores]

print(f"Students      : {len(scores)}")
print(f"Final avg     : {statistics.mean(finals):.1f}")
print(f"Final median  : {statistics.median(finals):.1f}")
print(f"Final stdev   : {statistics.stdev(finals):.1f}")
print(f"Top student   : {max(scores, key=lambda r: r['final'])['name']}")
print(f"At-risk (<75) : {[r['name'] for r in scores if r['final'] < 75]}")

grade_dist = {}
for r in scores:
    grade_dist[r["grade"]] = grade_dist.get(r["grade"], 0) + 1
print(f"\nGrade distribution: {dict(sorted(grade_dist.items()))}")

## End of Demo Notebook

For the full project report, see `report/final_report.md`.